# Full Pipeline: VNG Sentiment Analysis
Tất cả bộ source code Python nguyên gốc (`data_preprocessing.py`, `ml_models.py`, `dl_models.py`, `visualizations.py`) đã được bê nguyên vẹn vào Notebook này theo từng module chức năng để bảo quản đầy đủ kỹ thuật cốt lõi (Cross-Validation, Architecture, SMOTE).

## BƯỚC 1: IMPORT THƯ VIỆN CHUNG

In [ ]:
# 1. Import Thư Viện Tổng Hợp
import os
import re
import json
import time
import pickle
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout, SpatialDropout1D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except ImportError:
    HAS_UNDERTHESEA = False

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## BƯỚC 2: HÀM TIỀN XỬ LÝ DỮ LIỆU VÀ LÀM SẠCH VĂN BẢN
(Giữ nguyên 100% logic regex, stopwords, tokenization)

In [ ]:
"""
Data Preprocessing for VNG App Reviews Sentiment Analysis
- Load and clean review data
- Vietnamese text preprocessing
- Create sentiment labels from star ratings
- Train/test split
"""

# Try to import Vietnamese NLP tools
try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except ImportError:
    HAS_UNDERTHESEA = False
    print("[WARNING] underthesea not available. Using basic tokenization.")

# Vietnamese stopwords (common list)
VIETNAMESE_STOPWORDS = set([
    'và', 'của', 'có', 'là', 'cho', 'với', 'các', 'được', 'trong', 'không',
    'này', 'một', 'những', 'để', 'đã', 'nhưng', 'từ', 'khi', 'cũng', 'theo',
    'đến', 'về', 'như', 'hay', 'hoặc', 'tại', 'bị', 'nên', 'rất', 'thì',
    'đó', 'vì', 'mà', 'do', 'nó', 'ra', 'lại', 'còn', 'bởi', 'nếu',
    'ở', 'lên', 'xuống', 'vào', 'trên', 'dưới', 'qua', 'sau', 'trước',
    'ai', 'gì', 'nào', 'đâu', 'bao', 'sao', 'thế', 'vậy', 'này', 'kia',
    'ạ', 'à', 'ơi', 'ừ', 'nhé', 'nha', 'hen', 'nghen', 'á', 'ha',
    'tôi', 'tao', 'mình', 'ta', 'chúng', 'bạn', 'anh', 'chị', 'em',
    'nó', 'họ', 'ông', 'bà', 'cô', 'chú', 'thầy', 'cô',
    'the', 'and', 'is', 'in', 'it', 'of', 'to', 'a', 'an', 'for',
    'ok', 'oke', 'okey', 'okay',
])

def clean_text(text):
    """Clean and normalize Vietnamese text."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    text = text.lower().strip()

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove emojis and special unicode
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)
    # Remove numbers (standalone)
    text = re.sub(r'\b\d+\b', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def tokenize_text(text):
    """Tokenize Vietnamese text."""
    if not text:
        return ""

    if HAS_UNDERTHESEA:
        try:
            tokens = word_tokenize(text, format="text")
            return tokens
        except Exception:
            pass

    # Fallback: simple split
    return text

def remove_stopwords(text):
    """Remove Vietnamese and English stopwords."""
    if not text:
        return ""
    words = text.split()
    words = [w for w in words if w not in VIETNAMESE_STOPWORDS and len(w) > 1]
    return ' '.join(words)

def create_sentiment_label(score):
    """Convert star rating to sentiment label.
    1-2 stars -> 0 (Negative)
    3 stars   -> 1 (Neutral)
    4-5 stars -> 2 (Positive)
    """
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

def get_sentiment_name(label):
    """Get sentiment name from label."""
    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping.get(label, 'Unknown')

def load_and_preprocess(data_dir, output_dir):
    """Main preprocessing pipeline."""
    os.makedirs(output_dir, exist_ok=True)

    print("=" * 60)
    print("DATA PREPROCESSING PIPELINE")
    print("=" * 60)

    # Step 1: Load data
    print("\n[1/6] Loading data...")
    csv_files = [f for f in os.listdir(data_dir) if f.startswith('all_vng_reviews') and f.endswith('.csv')]
    if not csv_files:
        # Try individual app files
        csv_files = [f for f in os.listdir(data_dir) if f.endswith('_reviews.csv')]

    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(os.path.join(data_dir, f), encoding='utf-8-sig')
            dfs.append(df)
            print(f"   Loaded: {f} ({len(df)} rows)")
        except Exception as e:
            print(f"   Error loading {f}: {e}")

    if not dfs:
        raise ValueError("No data files found!")

    df = pd.concat(dfs, ignore_index=True)

    # Remove duplicates based on review_id
    initial_count = len(df)
    df = df.drop_duplicates(subset=['review_id'], keep='first')
    print(f"   Total: {initial_count} -> {len(df)} (removed {initial_count - len(df)} duplicates)")

    # Step 2: Filter valid reviews
    print("\n[2/6] Filtering valid reviews...")
    df = df[df['content'].notna() & (df['content'].str.strip() != '')]
    df = df[df['score'].notna()]
    df['score'] = df['score'].astype(int)
    print(f"   Valid reviews: {len(df)}")

    # Step 3: Create labels
    print("\n[3/6] Creating sentiment labels...")
    df['sentiment'] = df['score'].apply(create_sentiment_label)
    df['sentiment_name'] = df['sentiment'].apply(get_sentiment_name)

    label_counts = df['sentiment_name'].value_counts()
    print(f"   Label distribution:")
    for label, count in label_counts.items():
        print(f"     {label}: {count} ({count/len(df)*100:.1f}%)")

    # Step 4: Text preprocessing
    print("\n[4/6] Preprocessing text...")
    df['clean_text'] = df['content'].apply(clean_text)
    df['tokenized_text'] = df['clean_text'].apply(tokenize_text)
    df['processed_text'] = df['tokenized_text'].apply(remove_stopwords)

    # Remove empty processed texts
    df = df[df['processed_text'].str.strip() != '']
    print(f"   After text cleaning: {len(df)} reviews")

    # Add text length feature
    df['text_length'] = df['processed_text'].apply(lambda x: len(x.split()))

    # Step 5: Train/Test split
    print("\n[5/6] Splitting data (80/20)...")
    X = df['processed_text'].values
    y = df['sentiment'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"   Train: {len(X_train)}, Test: {len(X_test)}")
    print(f"   Train distribution: {dict(Counter(y_train))}")
    print(f"   Test distribution: {dict(Counter(y_test))}")

    # Step 6: Save processed data
    print("\n[6/6] Saving processed data...")

    # Save full processed dataframe
    df.to_csv(os.path.join(output_dir, 'processed_reviews.csv'),
              index=False, encoding='utf-8-sig')

    # Save train/test splits
    np.savez(os.path.join(output_dir, 'train_test_split.npz'),
             X_train=X_train, X_test=X_test,
             y_train=y_train, y_test=y_test)

    # Save metadata
    metadata = {
        'total_reviews': len(df),
        'train_size': len(X_train),
        'test_size': len(X_test),
        'num_classes': 3,
        'class_names': ['Negative', 'Neutral', 'Positive'],
        'label_distribution': {
            'Negative': int((y == 0).sum()),
            'Neutral': int((y == 1).sum()),
            'Positive': int((y == 2).sum()),
        },
        'apps': df['app_name'].unique().tolist() if 'app_name' in df.columns else [],
        'avg_text_length': float(df['text_length'].mean()),
        'has_underthesea': HAS_UNDERTHESEA,
    }

    with open(os.path.join(output_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"\n   Saved to: {output_dir}")
    print("   Files: processed_reviews.csv, train_test_split.npz, metadata.json")
    print("\nDone!")

    return df, X_train, X_test, y_train, y_test, metadata

## BƯỚC 3: HÀM MÔ HÌNH HÓA MACHINE LEARNING TIÊU CHUẨN
(Khai báo TF-IDF, SVM, LogisticRegression, Naive Bayes, Random Forest, Tính K-Fold CV)

In [ ]:
"""
Machine Learning Models for VNG App Reviews Sentiment Analysis
- TF-IDF Feature Extraction
- Naive Bayes, Logistic Regression, SVM, Random Forest
- Cross-Validation (StratifiedKFold k=5)
- Evaluation metrics
"""

    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)

CLASS_NAMES = ['Negative', 'Neutral', 'Positive']

def create_tfidf_features(X_train, X_test, max_features=10000):
    """Create TF-IDF features from text."""
    print("\n[TF-IDF] Creating features...")
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),  # unigram + bigram
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    print(f"   Feature shape: {X_train_tfidf.shape}")
    print(f"   Vocabulary size: {len(vectorizer.vocabulary_)}")

    return X_train_tfidf, X_test_tfidf, vectorizer

def get_models():
    """Get dictionary of ML models."""
    return {
        'Naive Bayes': MultinomialNB(alpha=1.0),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, C=1.0, solver='lbfgs',
            random_state=42, n_jobs=-1
        ),
        'SVM (Linear)': LinearSVC(
            max_iter=2000, C=1.0, random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=None,
            random_state=42, n_jobs=-1
        ),
    }

def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate a trained model."""
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision_macro': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
        'precision_weighted': float(precision_score(y_test, y_pred, average='weighted', zero_division=0)),
        'recall_weighted': float(recall_score(y_test, y_pred, average='weighted', zero_division=0)),
        'f1_weighted': float(f1_score(y_test, y_pred, average='weighted', zero_division=0)),
    }

    cm = confusion_matrix(y_test, y_pred).tolist()
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

    # Try ROC AUC (need probability estimates)
    try:
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)
        elif hasattr(model, 'decision_function'):
            y_scores = model.decision_function(X_test)
            # Convert to probabilities-like scores
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            y_proba = scaler.fit_transform(y_scores)
        else:
            y_proba = None

        if y_proba is not None:
            y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
            roc_auc = float(roc_auc_score(y_test_bin, y_proba, multi_class='ovr', average='macro'))
            metrics['roc_auc'] = roc_auc
    except Exception:
        pass

    return {
        'model_name': model_name,
        'metrics': metrics,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': y_pred.tolist(),
    }

def cross_validate_models(X_train_tfidf, y_train, n_folds=5):
    """Perform cross-validation for all models."""
    print(f"\n[Cross-Validation] StratifiedKFold k={n_folds}")
    print("-" * 60)

    models = get_models()
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    cv_results = {}

    scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

    for name, model in models.items():
        print(f"\n   {name}...")
        start_time = time.time()

        try:
            scores = cross_validate(
                model, X_train_tfidf, y_train,
                cv=cv, scoring=scoring,
                return_train_score=True, n_jobs=-1
            )

            elapsed = time.time() - start_time

            cv_results[name] = {
                'test_accuracy': scores['test_accuracy'].tolist(),
                'test_precision': scores['test_precision_macro'].tolist(),
                'test_recall': scores['test_recall_macro'].tolist(),
                'test_f1': scores['test_f1_macro'].tolist(),
                'train_accuracy': scores['train_accuracy'].tolist(),
                'mean_test_accuracy': float(scores['test_accuracy'].mean()),
                'std_test_accuracy': float(scores['test_accuracy'].std()),
                'mean_test_f1': float(scores['test_f1_macro'].mean()),
                'std_test_f1': float(scores['test_f1_macro'].std()),
                'time_seconds': elapsed,
            }

            print(f"     Accuracy: {scores['test_accuracy'].mean():.4f} (+/- {scores['test_accuracy'].std():.4f})")
            print(f"     F1 Score: {scores['test_f1_macro'].mean():.4f} (+/- {scores['test_f1_macro'].std():.4f})")
            print(f"     Time: {elapsed:.1f}s")

        except Exception as e:
            print(f"     ERROR: {e}")
            cv_results[name] = {'error': str(e)}

    return cv_results

def train_and_evaluate_all(X_train, X_test, y_train, y_test, output_dir):
    """Train all ML models and evaluate."""
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "=" * 60)
    print("MACHINE LEARNING PIPELINE")
    print("=" * 60)

    # Step 1: TF-IDF
    X_train_tfidf, X_test_tfidf, vectorizer = create_tfidf_features(X_train, X_test)

    # Save vectorizer
    with open(os.path.join(output_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
        pickle.dump(vectorizer, f)

    # Step 2: Cross-Validation
    cv_results = cross_validate_models(X_train_tfidf, y_train)

    # Step 3: Train and Evaluate on Test Set
    print("\n" + "=" * 60)
    print("TRAINING & EVALUATION ON TEST SET")
    print("=" * 60)

    models = get_models()
    all_results = {}
    trained_models = {}

    for name, model in models.items():
        print(f"\n--- {name} ---")
        start_time = time.time()

        model.fit(X_train_tfidf, y_train)
        train_time = time.time() - start_time

        result = evaluate_model(model, X_test_tfidf, y_test, name)
        result['train_time'] = train_time
        result['cv_results'] = cv_results.get(name, {})

        all_results[name] = result
        trained_models[name] = model

        print(f"   Accuracy:  {result['metrics']['accuracy']:.4f}")
        print(f"   F1 Macro:  {result['metrics']['f1_macro']:.4f}")
        print(f"   Precision: {result['metrics']['precision_macro']:.4f}")
        print(f"   Recall:    {result['metrics']['recall_macro']:.4f}")
        if 'roc_auc' in result['metrics']:
            print(f"   ROC AUC:   {result['metrics']['roc_auc']:.4f}")
        print(f"   Train time: {train_time:.2f}s")

    # Step 4: Get feature importance / top words
    print("\n[Feature Importance] Top words per class...")
    feature_names = vectorizer.get_feature_names_out()
    feature_importance = {}

    # From Logistic Regression coefficients
    lr_model = trained_models.get('Logistic Regression')
    if lr_model is not None and hasattr(lr_model, 'coef_'):
        for i, class_name in enumerate(CLASS_NAMES):
            if i < lr_model.coef_.shape[0]:
                top_indices = lr_model.coef_[i].argsort()[-20:][::-1]
                top_words = [(feature_names[j], float(lr_model.coef_[i][j])) for j in top_indices]
                feature_importance[class_name] = top_words
                print(f"   {class_name}: {', '.join([w for w, _ in top_words[:10]])}")

    # Step 5: Save results
    print("\n[Saving Results]...")

    # Save ML results
    results_save = {}
    for name, result in all_results.items():
        r = dict(result)
        r.pop('predictions', None)  # Don't save full predictions to JSON
        results_save[name] = r

    with open(os.path.join(output_dir, 'ml_results.json'), 'w', encoding='utf-8') as f:
        json.dump(results_save, f, ensure_ascii=False, indent=2, default=str)

    # Save CV results
    with open(os.path.join(output_dir, 'cv_results.json'), 'w', encoding='utf-8') as f:
        json.dump(cv_results, f, ensure_ascii=False, indent=2, default=str)

    # Save feature importance
    with open(os.path.join(output_dir, 'feature_importance.json'), 'w', encoding='utf-8') as f:
        json.dump(feature_importance, f, ensure_ascii=False, indent=2)

    # Save trained models
    for name, model in trained_models.items():
        safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
        with open(os.path.join(output_dir, f'model_{safe_name}.pkl'), 'wb') as f:
            pickle.dump(model, f)

    # Print summary table
    print("\n" + "=" * 60)
    print("ML RESULTS SUMMARY")
    print("=" * 60)
    print(f"\n{'Model':<25} {'Accuracy':>10} {'F1 Macro':>10} {'Precision':>10} {'Recall':>10}")
    print("-" * 65)
    for name, result in all_results.items():
        m = result['metrics']
        print(f"{name:<25} {m['accuracy']:>10.4f} {m['f1_macro']:>10.4f} "
              f"{m['precision_macro']:>10.4f} {m['recall_macro']:>10.4f}")

    return all_results, cv_results, trained_models, vectorizer, feature_importance

## BƯỚC 4: HÀM MÔ HÌNH HÓA ĐỘ CẤP SÂU (DEEP LEARNING)
(Khởi tạo Tokenizer Padding 80 ký tự, Build BiLSTM, CNN-1D, Callbacks)

In [ ]:
"""
Deep Learning Models for VNG App Reviews Sentiment Analysis
- LSTM (Bidirectional) and CNN (Conv1D) models
- Keras/TensorFlow based
- Cross-validation support
"""

sys.path.insert(0, r'C:\tf_pkg')

    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

CLASS_NAMES = ['Negative', 'Neutral', 'Positive']

def check_tensorflow():
    """Check if TensorFlow is available."""
    try:
        import tensorflow as tf
        print(f"   TensorFlow version: {tf.__version__}")
        return True
    except ImportError:
        print("   [WARNING] TensorFlow not available. Skipping DL models.")
        return False

def prepare_sequences(X_train, X_test, max_words=15000, max_len=100):
    """Tokenize and pad text sequences for DL models."""
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    print("\n[Sequence Preparation]")

    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

    vocab_size = min(max_words, len(tokenizer.word_index) + 1)
    print(f"   Vocab size: {vocab_size}")
    print(f"   Max sequence length: {max_len}")
    print(f"   Train shape: {X_train_pad.shape}")
    print(f"   Test shape: {X_test_pad.shape}")

    return X_train_pad, X_test_pad, tokenizer, vocab_size

def build_lstm_model(vocab_size, max_len, num_classes=3, embedding_dim=128):
    """Build Bidirectional LSTM model."""
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Embedding, Bidirectional, LSTM, Dense, Dropout,
        SpatialDropout1D, GlobalMaxPooling1D
    )

    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

def build_cnn_model(vocab_size, max_len, num_classes=3, embedding_dim=128):
    """Build CNN (Conv1D) model for text classification."""
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout,
        SpatialDropout1D, BatchNormalization
    )

    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        SpatialDropout1D(0.3),
        Conv1D(128, 5, activation='relu', padding='same'),
        BatchNormalization(),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

def train_dl_model(model, X_train, y_train, X_val, y_val, model_name,
                   epochs=20, batch_size=64):
    """Train a DL model and return history."""
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    print(f"\n--- Training {model_name} ---")
    print(f"   Epochs: {epochs}, Batch size: {batch_size}")

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )

    return history

def evaluate_dl_model(model, X_test, y_test, model_name):
    """Evaluate a DL model."""
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    metrics = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision_macro': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_test, y_pred, average='weighted', zero_division=0)),
    }

    cm = confusion_matrix(y_test, y_pred).tolist()
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

    print(f"\n   {model_name} Results:")
    print(f"   Accuracy:  {metrics['accuracy']:.4f}")
    print(f"   F1 Macro:  {metrics['f1_macro']:.4f}")
    print(f"   Precision: {metrics['precision_macro']:.4f}")
    print(f"   Recall:    {metrics['recall_macro']:.4f}")

    return {
        'model_name': model_name,
        'metrics': metrics,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': y_pred.tolist(),
    }

def cross_validate_dl(X_train_text, y_train, build_fn, model_name,
                      max_words=15000, max_len=100, n_folds=5,
                      epochs=15, batch_size=64):
    """Cross-validate a DL model."""
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    print(f"\n[DL Cross-Validation] {model_name} (k={n_folds})")

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_scores = {'accuracy': [], 'f1_macro': []}

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_text, y_train)):
        print(f"   Fold {fold+1}/{n_folds}...")

        X_fold_train = X_train_text[train_idx]
        X_fold_val = X_train_text[val_idx]
        y_fold_train = y_train[train_idx]
        y_fold_val = y_train[val_idx]

        # Tokenize
        tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
        tokenizer.fit_on_texts(X_fold_train)

        X_ft = pad_sequences(tokenizer.texts_to_sequences(X_fold_train),
                             maxlen=max_len, padding='post')
        X_fv = pad_sequences(tokenizer.texts_to_sequences(X_fold_val),
                             maxlen=max_len, padding='post')

        vocab_size = min(max_words, len(tokenizer.word_index) + 1)

        # Build and train
        model = build_fn(vocab_size, max_len)
        model.fit(X_ft, y_fold_train, validation_data=(X_fv, y_fold_val),
                  epochs=epochs, batch_size=batch_size, verbose=0,
                  callbacks=[__import__('tensorflow').keras.callbacks.EarlyStopping(
                      monitor='val_loss', patience=3, restore_best_weights=True)])

        # Evaluate
        y_pred = np.argmax(model.predict(X_fv, verbose=0), axis=1)
        acc = accuracy_score(y_fold_val, y_pred)
        f1 = f1_score(y_fold_val, y_pred, average='macro', zero_division=0)
        fold_scores['accuracy'].append(float(acc))
        fold_scores['f1_macro'].append(float(f1))
        print(f"     Accuracy: {acc:.4f}, F1: {f1:.4f}")

        # Clean up memory
        del model
        import gc
        gc.collect()

    fold_scores['mean_accuracy'] = float(np.mean(fold_scores['accuracy']))
    fold_scores['std_accuracy'] = float(np.std(fold_scores['accuracy']))
    fold_scores['mean_f1'] = float(np.mean(fold_scores['f1_macro']))
    fold_scores['std_f1'] = float(np.std(fold_scores['f1_macro']))

    print(f"   CV Accuracy: {fold_scores['mean_accuracy']:.4f} (+/- {fold_scores['std_accuracy']:.4f})")
    print(f"   CV F1:       {fold_scores['mean_f1']:.4f} (+/- {fold_scores['std_f1']:.4f})")

    return fold_scores

def train_and_evaluate_dl(X_train, X_test, y_train, y_test, output_dir,
                          max_words=15000, max_len=100, epochs=20, batch_size=64):
    """Train and evaluate all DL models."""
    os.makedirs(output_dir, exist_ok=True)

    if not check_tensorflow():
        print("TensorFlow not available. Skipping DL pipeline.")
        return {}, {}

    print("\n" + "=" * 60)
    print("DEEP LEARNING PIPELINE")
    print("=" * 60)

    # Step 1: Prepare sequences
    X_train_pad, X_test_pad, tokenizer, vocab_size = prepare_sequences(
        X_train, X_test, max_words, max_len
    )

    # Save tokenizer
    with open(os.path.join(output_dir, 'dl_tokenizer.pkl'), 'wb') as f:
        pickle.dump(tokenizer, f)

    # Step 2: Split train into train/val for training
    from sklearn.model_selection import train_test_split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_pad, y_train, test_size=0.15, random_state=42, stratify=y_train
    )

    # Step 3: Train models
    dl_models = {
        'BiLSTM': (build_lstm_model, {}),
        'CNN': (build_cnn_model, {}),
    }

    all_results = {}
    all_histories = {}
    dl_cv_results = {}

    for name, (build_fn, kwargs) in dl_models.items():
        print(f"\n{'='*50}")
        print(f"Training: {name}")
        print(f"{'='*50}")

        # Build model
        model = build_fn(vocab_size, max_len, **kwargs)
        model.summary()

        # Train
        history = train_dl_model(model, X_tr, y_tr, X_val, y_val,
                                 name, epochs=epochs, batch_size=batch_size)

        # Evaluate on test set
        result = evaluate_dl_model(model, X_test_pad, y_test, name)
        all_results[name] = result

        # Save history
        hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        all_histories[name] = hist

        # Save model
        safe_name = name.lower().replace(' ', '_')
        model.save(os.path.join(output_dir, f'dl_model_{safe_name}.keras'))

        # Cross-validation
        print(f"\n[Running CV for {name}...]")
        cv_result = cross_validate_dl(
            X_train, y_train, build_fn, name,
            max_words=max_words, max_len=max_len,
            n_folds=5, epochs=15, batch_size=batch_size
        )
        dl_cv_results[name] = cv_result

    # Step 4: Save results
    print("\n[Saving DL Results]...")

    results_save = {}
    for name, result in all_results.items():
        r = dict(result)
        r.pop('predictions', None)
        results_save[name] = r

    with open(os.path.join(output_dir, 'dl_results.json'), 'w', encoding='utf-8') as f:
        json.dump(results_save, f, ensure_ascii=False, indent=2, default=str)

    with open(os.path.join(output_dir, 'dl_histories.json'), 'w', encoding='utf-8') as f:
        json.dump(all_histories, f, ensure_ascii=False, indent=2)

    with open(os.path.join(output_dir, 'dl_cv_results.json'), 'w', encoding='utf-8') as f:
        json.dump(dl_cv_results, f, ensure_ascii=False, indent=2)

    # Print summary
    print("\n" + "=" * 60)
    print("DL RESULTS SUMMARY")
    print("=" * 60)
    print(f"\n{'Model':<20} {'Accuracy':>10} {'F1 Macro':>10} {'Precision':>10} {'Recall':>10}")
    print("-" * 60)
    for name, result in all_results.items():
        m = result['metrics']
        print(f"{name:<20} {m['accuracy']:>10.4f} {m['f1_macro']:>10.4f} "
              f"{m['precision_macro']:>10.4f} {m['recall_macro']:>10.4f}")

    return all_results, all_histories, dl_cv_results

## BƯỚC 5: HÀM VẼ TẤT CẢ BIỂU ĐỒ (VISUALIZATIONS)
(Confusion Matrices, ROC, Training Histories, CrossVal Boxplots)

In [ ]:
"""
Visualization module for VNG Sentiment Analysis
- ML and DL charts
- Confusion matrices, ROC curves, training curves
- Cross-validation box plots
- Data distribution charts
"""



# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

CLASS_NAMES = ['Negative', 'Neutral', 'Positive']
COLORS = {'Negative': '#e74c3c', 'Neutral': '#f39c12', 'Positive': '#2ecc71'}
MODEL_COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c']

def plot_data_distribution(df, output_dir):
    """Plot data analysis charts."""
    os.makedirs(output_dir, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('VNG App Reviews - Data Analysis', fontsize=18, fontweight='bold', y=1.02)

    # 1. Sentiment Distribution
    ax = axes[0, 0]
    sentiment_counts = df['sentiment_name'].value_counts()
    colors = [COLORS.get(s, '#999') for s in sentiment_counts.index]
    bars = ax.bar(sentiment_counts.index, sentiment_counts.values, color=colors, edgecolor='white', linewidth=1.5)
    for bar, count in zip(bars, sentiment_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                f'{count:,}\n({count/len(df)*100:.1f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count')

    # 2. Rating Distribution
    ax = axes[0, 1]
    rating_counts = df['score'].value_counts().sort_index()
    star_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#27ae60', '#2ecc71']
    ax.bar(rating_counts.index, rating_counts.values, color=star_colors, edgecolor='white', linewidth=1.5)
    for i, (rating, count) in enumerate(rating_counts.items()):
        ax.text(rating, count + 50, f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_title('Star Rating Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Stars')
    ax.set_ylabel('Count')
    ax.set_xticks(range(1, 6))

    # 3. Reviews per App
    ax = axes[1, 0]
    if 'app_name' in df.columns:
        app_counts = df['app_name'].value_counts()
        bars = ax.barh(app_counts.index, app_counts.values, color=MODEL_COLORS[:len(app_counts)],
                       edgecolor='white', linewidth=1)
        for bar, count in zip(bars, app_counts.values):
            ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2.,
                    f'{count:,}', va='center', fontweight='bold', fontsize=10)
        ax.set_title('Reviews per App', fontsize=14, fontweight='bold')
        ax.set_xlabel('Count')
    else:
        ax.text(0.5, 0.5, 'No app data', ha='center', va='center', transform=ax.transAxes)

    # 4. Text Length Distribution
    ax = axes[1, 1]
    if 'text_length' in df.columns:
        for sentiment in CLASS_NAMES:
            subset = df[df['sentiment_name'] == sentiment]['text_length']
            if len(subset) > 0:
                ax.hist(subset, bins=50, alpha=0.5, label=sentiment,
                        color=COLORS[sentiment], edgecolor='white')
        ax.set_title('Text Length Distribution by Sentiment', fontsize=14, fontweight='bold')
        ax.set_xlabel('Number of Words')
        ax.set_ylabel('Count')
        ax.legend(fontsize=11)
        ax.set_xlim(0, min(100, df['text_length'].quantile(0.99)))

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'data_distribution.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   Saved: data_distribution.png")

def plot_sentiment_by_app(df, output_dir):
    """Plot sentiment distribution per app."""
    if 'app_name' not in df.columns:
        return

    os.makedirs(output_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(14, 7))

    apps = df['app_name'].unique()
    x = np.arange(len(apps))
    width = 0.25

    for i, sentiment in enumerate(CLASS_NAMES):
        counts = [len(df[(df['app_name'] == app) & (df['sentiment_name'] == sentiment)]) for app in apps]
        ax.bar(x + i * width, counts, width, label=sentiment, color=COLORS[sentiment],
               edgecolor='white', linewidth=1)

    ax.set_xlabel('App', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Sentiment Distribution per VNG App', fontsize=16, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(apps, rotation=45, ha='right', fontsize=10)
    ax.legend(fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sentiment_by_app.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   Saved: sentiment_by_app.png")

def plot_confusion_matrices(results, output_dir, prefix='ml'):
    """Plot confusion matrices for all models."""
    os.makedirs(output_dir, exist_ok=True)

    n_models = len(results)
    cols = min(2, n_models)
    rows = (n_models + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 7 * rows))
    if n_models == 1:
        axes = [axes]
    elif rows > 1:
        axes = axes.flatten()

    for idx, (name, result) in enumerate(results.items()):
        ax = axes[idx] if n_models > 1 else axes[0]
        cm = np.array(result['confusion_matrix'])

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    ax=ax, cbar_kws={'shrink': 0.8},
                    annot_kws={'size': 14, 'fontweight': 'bold'})
        ax.set_title(f'{name}', fontsize=14, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=12)
        ax.set_xlabel('Predicted Label', fontsize=12)

    # Hide extra axes
    for idx in range(n_models, rows * cols):
        if n_models > 1:
            axes[idx].set_visible(False)

    plt.suptitle(f'Confusion Matrices ({prefix.upper()})', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{prefix}_confusion_matrices.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   Saved: {prefix}_confusion_matrices.png")

def plot_model_comparison(results, output_dir, prefix='ml'):
    """Plot model comparison bar charts."""
    os.makedirs(output_dir, exist_ok=True)

    models = list(results.keys())
    metrics_list = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
    metric_labels = ['Accuracy', 'F1 Score (Macro)', 'Precision (Macro)', 'Recall (Macro)']

    fig, axes = plt.subplots(1, len(metrics_list), figsize=(6 * len(metrics_list), 6))

    for i, (metric, label) in enumerate(zip(metrics_list, metric_labels)):
        ax = axes[i]
        values = [results[m]['metrics'].get(metric, 0) for m in models]
        bars = ax.bar(range(len(models)), values, color=MODEL_COLORS[:len(models)],
                      edgecolor='white', linewidth=1.5)

        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

        ax.set_title(label, fontsize=13, fontweight='bold')
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.axhline(y=max(values), color='gray', linestyle='--', alpha=0.3)

    plt.suptitle(f'Model Comparison ({prefix.upper()})', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{prefix}_model_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   Saved: {prefix}_model_comparison.png")

def plot_cv_boxplot(cv_results, output_dir, prefix='ml'):
    """Plot cross-validation box plots."""
    os.makedirs(output_dir, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Accuracy box plot
    ax = axes[0]
    data_acc = []
    labels = []
    for name, cv in cv_results.items():
        if 'test_accuracy' in cv:
            data_acc.append(cv['test_accuracy'])
            labels.append(name)
        elif 'accuracy' in cv:
            data_acc.append(cv['accuracy'])
            labels.append(name)

    if data_acc:
        bp = ax.boxplot(data_acc, labels=labels, patch_artist=True, widths=0.6)
        for patch, color in zip(bp['boxes'], MODEL_COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title('Cross-Validation Accuracy', fontsize=14, fontweight='bold')
        ax.set_ylabel('Accuracy', fontsize=12)
        ax.tick_params(axis='x', rotation=45)

    # F1 box plot
    ax = axes[1]
    data_f1 = []
    labels_f1 = []
    for name, cv in cv_results.items():
        if 'test_f1' in cv:
            data_f1.append(cv['test_f1'])
            labels_f1.append(name)
        elif 'f1_macro' in cv:
            data_f1.append(cv['f1_macro'])
            labels_f1.append(name)

    if data_f1:
        bp = ax.boxplot(data_f1, labels=labels_f1, patch_artist=True, widths=0.6)
        for patch, color in zip(bp['boxes'], MODEL_COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title('Cross-Validation F1 Score (Macro)', fontsize=14, fontweight='bold')
        ax.set_ylabel('F1 Score', fontsize=12)
        ax.tick_params(axis='x', rotation=45)

    plt.suptitle(f'Cross-Validation Results ({prefix.upper()}, k=5)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{prefix}_cv_boxplot.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   Saved: {prefix}_cv_boxplot.png")

def plot_dl_training_history(histories, output_dir):
    """Plot DL training/validation loss and accuracy curves."""
    os.makedirs(output_dir, exist_ok=True)

    for name, history in histories.items():
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss
        ax = axes[0]
        ax.plot(history['loss'], label='Train Loss', color='#3498db', linewidth=2)
        ax.plot(history['val_loss'], label='Val Loss', color='#e74c3c', linewidth=2)
        ax.set_title(f'{name} - Loss', fontsize=14, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=11)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

        # Accuracy
        ax = axes[1]
        ax.plot(history['accuracy'], label='Train Accuracy', color='#3498db', linewidth=2)
        ax.plot(history['val_accuracy'], label='Val Accuracy', color='#e74c3c', linewidth=2)
        ax.set_title(f'{name} - Accuracy', fontsize=14, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=11)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

        plt.suptitle(f'{name} Training History', fontsize=16, fontweight='bold')
        plt.tight_layout()
        safe_name = name.lower().replace(' ', '_')
        plt.savefig(os.path.join(output_dir, f'dl_training_{safe_name}.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   Saved: dl_training_{safe_name}.png")

def plot_all_models_comparison(ml_results, dl_results, output_dir):
    """Compare all ML and DL models together."""
    os.makedirs(output_dir, exist_ok=True)

    all_results = {}
    for name, r in ml_results.items():
        all_results[f"ML: {name}"] = r['metrics']
    for name, r in dl_results.items():
        all_results[f"DL: {name}"] = r['metrics']

    models = list(all_results.keys())
    metrics = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

    fig, ax = plt.subplots(figsize=(16, 8))

    x = np.arange(len(models))
    width = 0.2

    for i, (metric, label) in enumerate(zip(metrics, ['Accuracy', 'F1', 'Precision', 'Recall'])):
        values = [all_results[m].get(metric, 0) for m in models]
        bars = ax.bar(x + i * width, values, width, label=label,
                      color=MODEL_COLORS[i], edgecolor='white', linewidth=1)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Score', fontsize=13)
    ax.set_title('All Models Comparison (ML vs DL)', fontsize=18, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=12, loc='upper right')
    ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.3, label='Baseline')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'all_models_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   Saved: all_models_comparison.png")

def plot_classification_reports(results, output_dir, prefix='ml'):
    """Plot detailed classification report as heatmap."""
    os.makedirs(output_dir, exist_ok=True)

    for name, result in results.items():
        report = result.get('classification_report', {})
        if not report:
            continue

        # Extract per-class metrics
        classes = CLASS_NAMES
        metrics = ['precision', 'recall', 'f1-score']

        data = []
        for cls in classes:
            if cls in report:
                row = [report[cls].get(m, 0) for m in metrics]
            else:
                row = [0, 0, 0]
            data.append(row)

        df_report = pd.DataFrame(data, index=classes, columns=['Precision', 'Recall', 'F1-Score'])

        fig, ax = plt.subplots(figsize=(8, 4))
        sns.heatmap(df_report, annot=True, fmt='.3f', cmap='YlGnBu',
                    vmin=0, vmax=1, ax=ax, cbar_kws={'shrink': 0.8},
                    annot_kws={'size': 14, 'fontweight': 'bold'})
        ax.set_title(f'{name} - Classification Report', fontsize=14, fontweight='bold')

        plt.tight_layout()
        safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
        plt.savefig(os.path.join(output_dir, f'{prefix}_report_{safe_name}.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()

    print(f"   Saved: {prefix}_report_*.png")

def plot_wordcloud_per_sentiment(df, output_dir):
    """Plot word clouds for each sentiment class."""
    os.makedirs(output_dir, exist_ok=True)

    try:
        from wordcloud import WordCloud
    except ImportError:
        print("   [WARNING] wordcloud not installed. Skipping word clouds.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for i, sentiment in enumerate(CLASS_NAMES):
        ax = axes[i]
        text_col = 'processed_text' if 'processed_text' in df.columns else 'clean_text'
        if text_col not in df.columns:
            text_col = 'content'

        texts = df[df['sentiment_name'] == sentiment][text_col].dropna()
        all_text = ' '.join(texts.values)

        if len(all_text.strip()) < 10:
            ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center')
            ax.set_title(sentiment)
            continue

        wc = WordCloud(
            width=800, height=400,
            background_color='white',
            max_words=100,
            colormap='viridis' if sentiment == 'Neutral' else (
                'Reds' if sentiment == 'Negative' else 'Greens'),
            random_state=42,
        ).generate(all_text)

        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'{sentiment} Reviews', fontsize=14, fontweight='bold',
                     color=COLORS[sentiment])

    plt.suptitle('Word Clouds by Sentiment', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'wordclouds.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   Saved: wordclouds.png")

def generate_all_visualizations(output_dir):
    """Generate all visualizations from saved results."""
    print("\n" + "=" * 60)
    print("GENERATING VISUALIZATIONS")
    print("=" * 60)

    charts_dir = os.path.join(output_dir, 'charts')
    os.makedirs(charts_dir, exist_ok=True)

    # Load data
    processed_csv = os.path.join(output_dir, 'processed_reviews.csv')
    if os.path.exists(processed_csv):
        df = pd.read_csv(processed_csv, encoding='utf-8-sig')
        print("\n[1] Data Distribution Charts...")
        plot_data_distribution(df, charts_dir)
        plot_sentiment_by_app(df, charts_dir)
        print("\n[2] Word Clouds...")
        plot_wordcloud_per_sentiment(df, charts_dir)
    else:
        print("   No processed_reviews.csv found, skipping data charts.")
        df = None

    # ML Results
    ml_results_path = os.path.join(output_dir, 'ml_results.json')
    ml_results = {}
    if os.path.exists(ml_results_path):
        with open(ml_results_path, 'r', encoding='utf-8') as f:
            ml_results = json.load(f)

        print("\n[3] ML Confusion Matrices...")
        plot_confusion_matrices(ml_results, charts_dir, prefix='ml')

        print("\n[4] ML Model Comparison...")
        plot_model_comparison(ml_results, charts_dir, prefix='ml')

        print("\n[5] ML Classification Reports...")
        plot_classification_reports(ml_results, charts_dir, prefix='ml')

    # ML CV Results
    cv_results_path = os.path.join(output_dir, 'cv_results.json')
    if os.path.exists(cv_results_path):
        with open(cv_results_path, 'r', encoding='utf-8') as f:
            cv_results = json.load(f)
        print("\n[6] ML Cross-Validation Box Plots...")
        plot_cv_boxplot(cv_results, charts_dir, prefix='ml')

    # DL Results
    dl_results_path = os.path.join(output_dir, 'dl_results.json')
    dl_results = {}
    if os.path.exists(dl_results_path):
        with open(dl_results_path, 'r', encoding='utf-8') as f:
            dl_results = json.load(f)

        print("\n[7] DL Confusion Matrices...")
        plot_confusion_matrices(dl_results, charts_dir, prefix='dl')

        print("\n[8] DL Model Comparison...")
        plot_model_comparison(dl_results, charts_dir, prefix='dl')

        print("\n[9] DL Classification Reports...")
        plot_classification_reports(dl_results, charts_dir, prefix='dl')

    # DL Training History
    dl_hist_path = os.path.join(output_dir, 'dl_histories.json')
    if os.path.exists(dl_hist_path):
        with open(dl_hist_path, 'r', encoding='utf-8') as f:
            dl_histories = json.load(f)
        print("\n[10] DL Training History Curves...")
        plot_dl_training_history(dl_histories, charts_dir)

    # DL CV Results
    dl_cv_path = os.path.join(output_dir, 'dl_cv_results.json')
    if os.path.exists(dl_cv_path):
        with open(dl_cv_path, 'r', encoding='utf-8') as f:
            dl_cv = json.load(f)
        print("\n[11] DL Cross-Validation Box Plots...")
        plot_cv_boxplot(dl_cv, charts_dir, prefix='dl')

    # All models comparison
    if ml_results and dl_results:
        print("\n[12] All Models Comparison (ML vs DL)...")
        plot_all_models_comparison(ml_results, dl_results, charts_dir)

    print(f"\n All charts saved in: {charts_dir}")
    print("Done!")

## BƯỚC 6: CHẠY THỰC THI TOÀN BỘ PIPELINE (CÓSMOTE)
Nơi giao thoa tất cả các Object đã được biên dịch bên trên.

In [ ]:
# 1. LOAD DATA & TIỀN XỬ LÝ (Từ File CSV Gốc)
DATA_DIR = '../vng_reviews_data'
OUTPUT_DIR = './output'

print("--- [BƯỚC 1]: LOAD DATA ---")
# Hàm load_and_preprocess lấy trọn vẹn từ data_preprocessing.py
df, X_train, X_test, y_train, y_test, metadata = load_and_preprocess(DATA_DIR, OUTPUT_DIR)

# Vẽ Khám phá Dữ Liệu
plot_data_distribution(df, OUTPUT_DIR)


In [ ]:
# 2. XỬ LÝ MẤT CÂN BẰNG BẰNG SMOTE ĐỐI VỚI ML
print("--- [BƯỚC 2]: TẠO TF-IDF & CÂN BẰNG NHÃN BẰNG SMOTE ---")
X_train_tfidf, X_test_tfidf, vectorizer = create_tfidf_features(X_train, X_test)

if HAS_SMOTE:
    print(f"Phân bố gốc (Imbalance): {Counter(y_train)}")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)
    print(f"Phân bố sau SMOTE (Balanced): {Counter(y_train_resampled)}")
else:
    print("Không có SMOTE, sử dụng class weights")
    X_train_resampled, y_train_resampled = X_train_tfidf, y_train


In [ ]:
# 3. TRAINING MACHINE LEARNING models với SMOTE
# Do train_and_evaluate_all trong ml_models nhận X_train dạng text, ta tách ra gọi hàm fit thủ công để đưa Smote vào
print("--- [BƯỚC 3]: TRAINING MACHINE LEARNING MODELS ---")
models = get_models()
ml_results = {}

for name, model in models.items():
    print(f"\nĐào tạo: {name}")
    start_time = time.time()
    model.fit(X_train_resampled, y_train_resampled)
    
    # Eval
    result = evaluate_model(model, X_test_tfidf, y_test, name)
    result['train_time'] = time.time() - start_time
    ml_results[name] = result
    
    # In điểm
    m = result['metrics']
    print(f"Acc: {m['accuracy']:.4f} | F1: {m['f1_macro']:.4f} | Recall: {m['recall_macro']:.4f}")

# Vẽ Confusion Matrix ML
plot_confusion_matrices(ml_results, OUTPUT_DIR, prefix='ml')


In [ ]:
# 4. TRAINING DEEP LEARNING (BiLSTM & CNN)
print("--- [BƯỚC 4]: TRAINING DEEP LEARNING ---")
# Hàm train_and_evaluate_dl_all sẽ padding Sequences, compile Model keras và run EarlyStopping
# (Lấy trọn vẹn từ dl_models.py)
dl_results, dl_cv_results, dl_histories = train_and_evaluate_dl_all(X_train, X_test, y_train, y_test, OUTPUT_DIR)

# Biểu đồ Lịch sử Training DL
plot_dl_training_history(dl_histories, OUTPUT_DIR)
